<br>

# FDBS

Funções

<br>

Michel Metran\
Data: 02.11.2025\
Atualizado em: 02.11.2025


In [2]:
import asyncio
import concurrent.futures
import os
import time
from datetime import datetime, timedelta
from pathlib import Path
from urllib.parse import urljoin

import aiohttp
import pandas as pd
import pytz
import requests
import requests_cache  # Para cache das requisições
from lxml import html
from tqdm.notebook import tqdm  # para barra de progresso

<br>

---

## Funções


In [ ]:
def get_links(url, get_size=False):
    """
    Obtém todos os links de uma página e opcionalmente os tamanhos das pastas

    Parameters:
    -----------
    url : str
        URL da página
    get_size : bool
        Se True, tenta obter o tamanho das pastas
    """
    try:
        response = requests.get(url)
        response.raise_for_status()
        tree = html.fromstring(response.content)

        # Links a ignorar
        ignore_links = [
            'http://browsehappy.com',
            'https://larsjung.de/h5ai/',
        ]

        if get_size:
            # Obtém links e tamanhos usando a estrutura correta do HTML
            items = []
            folders = tree.xpath('//li[contains(@class, "item folder")]')
            folders = tree.xpath('//tr')

            for folder in folders:
                link = folder.xpath('.//a/@href')[0]

                if not any(ignore in link for ignore in ignore_links):
                    name = folder.xpath('.//span[@class="label"]/@title')[0]
                    date = folder.xpath('.//span[@class="date"]/@data-time')[0]
                    size_bytes = folder.xpath(
                        './/span[@class="size"]/@data-bytes'
                    )[0]
                    size_formatted = folder.xpath(
                        './/span[@class="size"]/text()'
                    )[0].strip()

                    items.append(
                        {
                            'link': link,
                            'name': name,
                            'date': date,
                            'size': size_formatted,
                            'size_bytes': int(size_bytes),
                        }
                    )
            return items

        else:
            # Apenas links (mantém o comportamento anterior para compatibilidade)
            links = tree.xpath('//a/@href')
            # Filtra links de navegação e links a ignorar
            links = [
                link
                for link in links
                if not link.startswith('../')
                and not any(ignore in link for ignore in ignore_links)
            ]
            return sorted(links)

    except Exception as e:
        print(f"Erro ao acessar {url}: {str(e)}")
        return []


def get_states():
    """Lista todos os estados disponíveis com seus tamanhos"""
    base_url = 'https://geo.fbds.org.br/'
    states_info = get_links(base_url, get_size=True)
    return [
        {
            'estado': item['name'],
            'url': urljoin(base_url, item['link']),
            'tamanho': item['size'],
            'tamanho_bytes': item['size_bytes'],
            'data': item['date'],
        }
        for item in states_info
    ]


def get_municipalities(state_url):
    """Lista todos os municípios de um estado com seus tamanhos"""
    municipalities_info = get_links(state_url, get_size=True)
    return [
        {
            'municipio': item['name'],
            'url': urljoin(state_url, item['link']),
            'tamanho': item['size'],
            'tamanho_bytes': item['size_bytes'],
            'data': item['date'],
        }
        for item in municipalities_info
    ]


def get_layers(municipality_url):
    """Lista todas as camadas de um município com seus tamanhos"""
    layers_info = get_links(municipality_url, get_size=True)
    return [
        {
            'layer': item['name'],
            'url': urljoin(municipality_url, item['link']),
            'tamanho': item['size'],
            'tamanho_bytes': item['size_bytes'],
            'data': item['date'],
        }
        for item in layers_info
    ]


def get_files(layer_url):
    """Lista todos os arquivos de uma camada com seus tamanhos"""
    files_info = get_links(layer_url, get_size=True)
    return [
        {
            'arquivo': item['name'],
            'url': urljoin(layer_url, item['link']),
            'tamanho': item['size'],
            'tamanho_bytes': item['size_bytes'],
            'data': item['date'],
        }
        for item in files_info
    ]

In [ ]:
# Lista todos os estados com informações detalhadas
print("Obtendo lista de estados...")
states_df = pd.DataFrame(get_states())
print(f"\nEstados encontrados: {len(states_df)}")

# Converte a coluna de data para datetime
states_df['data'] = pd.to_datetime(states_df['data'].astype(float), unit='ms')

# Ordena por tamanho
states_df = states_df.sort_values('tamanho_bytes', ascending=False)

# Mostra os resultados
print("\nEstados ordenados por tamanho:")
states_df

In [ ]:
# aa = '1517510720000'
# time.asctime()
# time.mktime(aa)
# O timestamp em milissegundos
timestamp_ms = 1517510720000

# 1. Converter para segundos (Float)
timestamp_s = timestamp_ms / 1000

# --- Opção 1: Data e Hora UTC (Recomendado para o valor base) ---
# Cria o objeto datetime com o fuso horário UTC
dt_utc = datetime.fromtimestamp(
    timestamp_s,
    # tz=pytz.utc,
    tz=pytz.timezone('America/Sao_Paulo'),
)

# --- Opção 2: Data e Hora no Fuso Horário Local (do seu computador) ---
# Cria o objeto datetime, mas o tz será o local do seu sistema
# dt_local = datetime.fromtimestamp(timestamp_s)

# --- Opção 3: Convertendo para um Fuso Horário Específico (Ex: São Paulo) ---
# Certifique-se de que dt_utc é o ponto de partida (naive) ou use o pytz para converter
# fuso_sp = pytz.timezone('America/Sao_Paulo')
# dt_sp = dt_utc.astimezone(fuso_sp)

# print(f"Timestamp original (ms): {timestamp_ms}")
# print("-" * 30)
print(f"1. Hora UTC: {dt_utc.strftime('%Y-%m-%d %H:%M:%S %Z')}")
# print(f"2. Hora Local: {dt_local.strftime('%Y-%m-%d %H:%M:%S')}")
# print(f"3. Hora em SP: {dt_sp.strftime('%Y-%m-%d %H:%M:%S %Z')}")

# Detalhe do timestamp 1517510720000:
# A data e hora exata em UTC é: 2018-02-01 19:25:20 UTC

In [ ]:
def collect_all_data(max_states=None, max_municipalities=None):
    """
    Coleta todos os dados do site FBDS

    Parameters:
    -----------
    max_states : int, optional
        Número máximo de estados para processar (para testes)
    max_municipalities : int, optional
        Número máximo de municípios por estado para processar (para testes)
    """
    all_data = []

    # Obtém estados
    states = get_states()
    if max_states:
        states = states[:max_states]

    # Para cada estado
    for state in states:
        print(f"\nProcessando estado: {state['estado']} ({state['tamanho']})")

        # Obtém municípios
        municipalities = get_municipalities(state['url'])
        if max_municipalities:
            municipalities = municipalities[:max_municipalities]

        # Para cada município
        for mun in municipalities:
            print(
                f"  Processando município: {mun['municipio']} ({mun['tamanho']})"
            )

            # Obtém camadas
            layers = get_layers(mun['url'])

            # Para cada camada
            for layer in layers:
                print(
                    f"    Processando camada: {layer['layer']} ({layer['tamanho']})"
                )

                # Obtém arquivos
                files = get_files(layer['url'])

                # Adiciona informações aos dados
                for file in files:
                    data = {
                        'estado': state['estado'],
                        'estado_tamanho': state['tamanho'],
                        'municipio': mun['municipio'],
                        'municipio_tamanho': mun['tamanho'],
                        'camada': layer['layer'],
                        'camada_tamanho': layer['tamanho'],
                        'arquivo': file['arquivo'],
                        'arquivo_tamanho': file['tamanho'],
                        'url': file['url'],
                    }
                    all_data.append(data)

                # Pequeno delay para não sobrecarregar o servidor
                time.sleep(0.5)

    # Converte para DataFrame
    df = pd.DataFrame(all_data)

    print("\nColeta concluída!")
    print(f"Total de arquivos encontrados: {len(df)}")

    return df

In [ ]:
async def download_file_async(session, url_info, output_dir, pbar):
    """
    Download assíncrono de um único arquivo
    """
    try:
        url = url_info['url']
        output_path = Path(output_dir) / url_info['name']

        # Cria o diretório se não existir
        output_path.parent.mkdir(parents=True, exist_ok=True)

        # Faz o download
        async with session.get(url) as response:
            if response.status == 200:
                content = await response.read()

                # Salva o arquivo
                with open(output_path, 'wb') as f:
                    f.write(content)

                result = {
                    'nome': url_info['name'],
                    'status': 'sucesso',
                }
            else:
                result = {
                    'nome': url_info['name'],
                    'status': 'erro',
                    'erro': f'Status code: {response.status}',
                }
    except Exception as e:
        result = {'nome': url_info['name'], 'status': 'erro', 'erro': str(e)}

    pbar.update(1)
    return result


async def download_files_async(url_list, output_dir, max_concurrent=5):
    """
    Download assíncrono de múltiplos arquivos

    Parameters:
    -----------
    url_list : list
        Lista de dicionários com informações dos arquivos
    output_dir : str or Path
        Diretório onde salvar os arquivos
    max_concurrent : int
        Número máximo de downloads simultâneos
    """
    # Configura conexão com limite de conexões simultâneas
    conn = aiohttp.TCPConnector(limit=max_concurrent)

    # Lista para armazenar resultados
    results = []

    # Cria a barra de progresso
    with tqdm(total=len(url_list), desc="Downloading files") as pbar:
        # Sessão assíncrona
        async with aiohttp.ClientSession(connector=conn) as session:
            # Cria as tasks
            tasks = [
                download_file_async(session, url_info, output_dir, pbar)
                for url_info in url_list
            ]

            # Executa as tasks com limite de concorrência
            results = await asyncio.gather(*tasks)

    return results


def download_files_wrapper(url_list, output_dir, max_concurrent=5):
    """
    Wrapper para executar o download assíncrono
    """
    # Cria um novo event loop
    loop = asyncio.new_event_loop()
    asyncio.set_event_loop(loop)

    try:
        # Executa o download assíncrono
        results = loop.run_until_complete(
            download_files_async(url_list, output_dir, max_concurrent)
        )
    finally:
        # Fecha o loop
        loop.close()

    return results

In [ ]:
# Salva os resultados em um arquivo CSV
output_file = 'fbds_files.csv'
df.to_csv(output_file, index=False)
print(f"\nDados salvos em {output_file}")

# Mostra algumas estatísticas
print("\nEstatísticas:")
print(f"Total de estados: {df['estado'].nunique()}")
print(f"Total de municípios: {df['municipio'].nunique()}")
print(f"Total de camadas: {df['camada'].nunique()}")
print(f"Total de arquivos: {len(df)}")

# Mostra as camadas disponíveis
print("\nCamadas disponíveis:")
print(df['camada'].unique())